Week 10 (30/04/2025) - TBA

Word stemming might result in problematic situations, such as "overstemming", which happens when two unrelated words should be reduced to the same root, or "understemming", which happens when two different words with the same root get reduced to different roots instead.

In [11]:
'''
nltk.download("averaged_perceptron_tagger_eng")
nltk.download('tagsets_json')
'''

import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tag import pos_tag

# Stemming consists in reducing a word to its root.
# Start by creating an instance of the stemming model.
stemmer = PorterStemmer()
words = "The crew of the USS Discovery discovered many discoveries. Discovering is what explorers do"
stemmed_words = [stemmer.stem(word) for word in words] # list comprehension performing the stemming task

# Try using part of speech tagging for stemming.
tags = nltk.pos_tag(stemmed_words)
print(tags) # associate each word to the corresponding tag
# print(nltk.help.upenn_tagset()) # to understand the meaning of each tag

# Alternatively, it is possible to prepare data by lemmatizing, which consists in getting the core meaning of a word.
lemmatizer = WordNetLemmatizer()
lemm = lemmatizer.lemmatize('scarves')
print(lemm)

x = lemmatizer.lemmatize('worst', pos='a') # to consider adjectives
print(x)

[('t', 'NN'), ('h', 'NN'), ('e', 'NN'), (' ', 'NNP'), ('c', 'VBZ'), ('r', 'NN'), ('e', 'NN'), ('w', 'NN'), (' ', 'NNP'), ('o', 'VBZ'), ('f', 'JJ'), (' ', 'NNP'), ('t', 'NN'), ('h', 'NN'), ('e', 'NN'), (' ', 'NNP'), ('u', 'JJ'), ('s', 'NN'), ('s', 'NN'), (' ', 'NNP'), ('d', 'NN'), ('i', 'NN'), ('s', 'VBP'), ('c', 'NN'), ('o', 'NN'), ('v', 'NN'), ('e', 'NN'), ('r', 'NN'), ('y', 'NN'), (' ', 'NNP'), ('d', 'NN'), ('i', 'NN'), ('s', 'VBP'), ('c', 'NN'), ('o', 'NN'), ('v', 'NN'), ('e', 'NN'), ('r', 'NN'), ('e', 'NN'), ('d', 'NN'), (' ', 'NNP'), ('m', 'VBZ'), ('a', 'DT'), ('n', 'JJ'), ('y', 'NN'), (' ', 'NNP'), ('d', 'NN'), ('i', 'NN'), ('s', 'VBP'), ('c', 'NN'), ('o', 'NN'), ('v', 'NN'), ('e', 'NN'), ('r', 'NN'), ('i', 'NN'), ('e', 'VBP'), ('s', 'NN'), ('.', '.'), (' ', 'JJ'), ('d', 'NN'), ('i', 'NN'), ('s', 'VBP'), ('c', 'NN'), ('o', 'NN'), ('v', 'NN'), ('e', 'NN'), ('r', 'NN'), ('i', 'NN'), ('n', 'VBP'), ('g', 'NN'), (' ', 'NN'), ('i', 'NN'), ('s', 'VBP'), (' ', 'JJ'), ('w', 'NN'), ('h', '

Feature engineering

In [12]:
from sklearn.feature_extraction import DictVectorizer

# Create a dataset, containing data about houses.
data = [
    {'price': 200000, 'rooms': 4, 'neighbourhood': 'Appia'},
    {'price': 140000, 'rooms': 3, 'neighbourhood': 'Tuscolana'},
    {'price': 120000, 'rooms': 2, 'neighbourhood': 'Casilina'}
]

# Since mixing data might be problematic, map the neighbourhoods onto IDs via a vectorizer.
neigh = {'Appia': 1, 'Tuscolana': 2, 'Casilina': 3}

# Start by creating an instance of the vectorizer model.
vec = DictVectorizer(sparse=False, dtype=int)
res = vec.fit_transform(data) # convert the numerical values to one-hot encoding
print(vec.get_feature_names_out()) # to print the association results

['neighbourhood=Appia' 'neighbourhood=Casilina' 'neighbourhood=Tuscolana'
 'price' 'rooms']


Word count exercise

In [15]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

sample = [
    'problem of evil',
    'evil queen',
    'horizon problem'
]

# Create an instance of the vectorizer model.
vec = CountVectorizer()
X = vec.fit_transform(sample)
print(X)

# The word count provides a more compact representation of the strings and the corresponding counts.
# Therefore, considering converting the output to a more suitable data frame.
x_pd = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
print(x_pd)
# The result is a table that counts the occurrence of each word in each sentence in the table

  (0, 3)	1
  (0, 2)	1
  (0, 0)	1
  (1, 0)	1
  (1, 4)	1
  (2, 3)	1
  (2, 1)	1
   evil  horizon  of  problem  queen
0     1        0   1        1      0
1     1        0   0        0      1
2     0        1   0        1      0


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

sample = [
    'problem of evil',
    'evil queen',
    'horizon problem'
]

# Create an instance of the vectorizer model with an alternative approach that weights the count.
vec = TfidfVectorizer()
X = vec.fit_transform(sample)
print(X)

# The word count provides a more compact representation of the strings and the corresponding counts.
# Therefore, considering converting the output to a more suitable data frame.
x_pd = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
print(x_pd)
# The result is a table that counts the occurrence of each word in each sentence in the table

  (0, 3)	0.5178561161676974
  (0, 2)	0.680918560398684
  (0, 0)	0.5178561161676974
  (1, 0)	0.6053485081062916
  (1, 4)	0.7959605415681652
  (2, 3)	0.6053485081062916
  (2, 1)	0.7959605415681652
       evil   horizon        of   problem     queen
0  0.517856  0.000000  0.680919  0.517856  0.000000
1  0.605349  0.000000  0.000000  0.000000  0.795961
2  0.000000  0.795961  0.000000  0.605349  0.000000


Training a model for text topics.

In [28]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Start by getting the data and use a subset of the categories.
categories = ['talk.religion.misc', 'sci.space', 'comp.graphics'] # define the categories
train = fetch_20newsgroups(subset='train', categories=categories) # get the training dataset
test = fetch_20newsgroups(subset='test', categories=categories) # get the testing dataset

# At this point, create the pipeline for the model by specifying the steps to perform.
model = make_pipeline(TfidfVectorizer(), MultinomialNB())

# Train the model by fitting the training dataset.
model.fit(train.data, train.target)

# Test the model on the new testing dataset.
labels = model.predict(test.data)

'''
# Plot the confusion matrix to spot eventual problematic topics.
conf_matrix = confusion_matrix(test.target, labels)
sns.heatmap(conf_matrix.T, fmt='d', square=True, cbar=False, annot=True, xticklabels=train.target_names, yticklabels=train.target_names)
plt.xlabel('true labels')
plt.ylabel('predicted labels')
plt.show()
'''

# Now, try to classify a new text.
# s = 'I believe I can fly'
# s = 'The hit of the light'
# s = 'In the name of God'
# s = 'Use the texture as you wish' # problematic for some reason
s = 'You have to do raytracing'
pred = model.predict([s]) # remember to put s in a table in order to avoid problems with Scikit-Learn
print(train.target_names[pred[0]]) # print the label associated to the string

comp.graphics


The Gaussian Naive Bayes classification model is a machine learning model that performs classification through posterior probabilities that are computed via Bayes' theorem as P(label|features), under the assumption that the sample data follows a Gaussian distribution. <br>
Most particularly, it is possible to generlize the algorithm to a Multinomial Naive Bayes method, which assumes that the sample data follows a multinomial distribution. <br>
Overall, while a Naive Bayes model is not very efficient, it is very fast and straightforward, therefore allowing for simple interpretations of the results.

Support vector machines is a supervised learning algorithm that aims to find a vector that splits data into the given classes. <br>
Generally speaking, the criterion for the algorithm consists in choosing the line that has the highest amount of space between itself and each class, meaning that it is able to separate them as much as possible. <br>
Keep in mind, however, that the algorithm easily struggles with classes that are not linearly separated